# 🍎 Chat Completions with AIProjectClient 🍏

In this notebook, we'll demonstrate how to perform **Chat Completions** using the **Azure AI Foundry** SDK. We'll combine **`azure-ai-projects`** and **`azure-ai-inference`** packages to:

1. **Initialize** an `AIProjectClient`.
2. **Obtain** a Chat Completions client to do direct LLM calls.
3. **Use** a **prompt template** to add system context.
4. **Send** user prompts in a health & fitness theme.

## 🏋️ Health-Fitness Disclaimer
> **This example is for demonstration only and does not provide real medical advice.** Always consult a professional for health or medical-related questions.

### Prerequisites
Before starting this notebook, please ensure you have completed all prerequisites listed in the root [README.md](../../README.md#-prerequisites).

Let's get started! 🎉

<img src="./seq-diagrams/1-chat.png" width="30%"/>


## 1. Initial Setup
Load environment variables, create an `AIProjectClient`, and fetch a `ChatCompletionsClient`. We'll also define a **prompt template** to show how you might structure a system message.


In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from urllib.parse import urlparse
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.inference.models import UserMessage, SystemMessage
import requests

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

# Initialize credentials using AzureCliCredential (works better in notebooks)
credential = AzureCliCredential()

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = path_parts[-1] if path_parts else ""
hub_name        = _parsed.netloc.split(".")[0]
model_deployment = os.getenv("MODEL_DEPLOYMENT_NAME")

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    # List all accessible subscriptions
    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None
    
    for sub in subs:
        sub_id = sub["subscriptionId"]
        # Search for the Foundry hub: type=Microsoft.CognitiveServices/accounts, name=hub_name
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            # ARM resource ID: /subscriptions/<sub>/resourceGroups/<rg>/providers/...
            rg_from_id = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            resource_group = rg_from_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any accessible subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(
        f"Failed to auto-detect subscription and resource group: {e}"
    ) from e

try:
    project_client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✅ Successfully created AIProjectClient")
except Exception as e:
    print("❌ Error initializing client:", e)

### Prompt Template
We'll define a quick **system** message that sets the context as a friendly, disclaimer-providing fitness assistant.

```txt
SYSTEM PROMPT (template):
You are FitChat GPT, a helpful fitness assistant.
Always remind users: I'm not a medical professional.
Be friendly, provide general advice.
...
```

We'll then pass user content as a **user** message.


In [ ]:
# We'll define a function that runs chat completions with a system prompt & user prompt
from openai import OpenAI

def chat_with_fitness_assistant(user_input: str):
    """Use chat completions to get a response from our LLM, with system instructions."""
    # Our system message template
    system_text = (
        "You are FitChat GPT, a friendly fitness assistant.\n"
        "Always remind users: I'm not a medical professional.\n"
        "Answer with empathy and disclaimers."
    )

    # Create OpenAI client using Foundry endpoint and API key
    openai_client = OpenAI(
        api_key=os.getenv("AZURE_OPENAI_KEY"),
        base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    )
    
    # Construct messages: system + user
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_input}
    ]

    # Send the request
    response = openai_client.chat.completions.create(
        model=model_deployment,
        messages=messages
    )

    return response.choices[0].message.content  # simplest approach: get top choice's content

print("Defined a helper function to do chat completions.")

## 2. Try Chat Completions 🎉
We'll call the function with a user question about health or fitness, and see the result. Feel free to modify the question or run multiple times!


In [ ]:
user_question = "How can I start a beginner workout routine at home?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ User:", user_question)
print("🤖 Assistant:", reply)

## 3. Another Example: Prompt Template with Fill-Ins 📝
We can go a bit further and add placeholders in the system message. For instance, imagine we have a **userName** or **goal**. We'll show a minimal example.


In [ ]:
def chat_with_template(user_input: str, user_name: str, goal: str):
    # Construct a system template with placeholders
    system_template = (
        "You are FitChat GPT, an AI personal trainer for {name}.\n"
        "Your user wants to achieve: {goal}.\n"
        "Remind them you're not a medical professional. Offer friendly advice."
    )

    # Fill in placeholders
    system_prompt = system_template.format(name=user_name, goal=goal)

    # Create OpenAI client using Foundry endpoint and API key
    openai_client = OpenAI(
        api_key=os.getenv("AZURE_OPENAI_KEY"),
        base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    )
    
    # Construct messages: system + user
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    # Send the request
    response = openai_client.chat.completions.create(
        model=model_deployment,
        messages=messages
    )

    return response.choices[0].message.content

# Let's try it out
templated_user_input = "What kind of home exercise do you recommend for a busy schedule?"
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="Jordan",
    goal="increase muscle tone and endurance"
)
print("🗣️ User:", templated_user_input)
print("🤖 Assistant:", assistant_reply)

## 🎉 Congratulations!
You've successfully performed **chat completions** with the Azure AI Foundry's `AIProjectClient` and `azure-ai-inference`. You've also seen how to incorporate **prompt templates** to tailor your system instructions.

#### Head to [2-embeddings.ipynb](2-embeddings.ipynb) for the next part of the workshop! 🎯